# Immune integration: Multimodal (RNA + ATAC) training

Train `RegularizedMultimodalVI` on 7-dataset immune integration.
Uses early stopping with stratified validation split.

**Input**: `adata_rna.h5ad` + `adata_atac_tiles.h5ad` + `scrublet_results.csv`
**Output**: Trained multimodal model + latent representations + UMAPs

In [ ]:
import scanpy as sc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import rcParams
import torch
import os
import mudata as mu

import scvi
import regularizedvi

scvi.settings.progress_bar_style = "tqdm"
torch.set_float32_matmul_precision("high")
rcParams["pdf.fonttype"] = 42

## Data loading

In [ ]:
data_folder = "results/immune_integration/"
rna_path = os.path.join(data_folder, "adata_rna.h5ad")
atac_path = os.path.join(data_folder, "adata_atac_tiles.h5ad")
scrublet_path = os.path.join(data_folder, "scrublet_results.csv")

adata_rna = sc.read_h5ad(rna_path)
print(f"RNA: {adata_rna.shape}")

adata_atac = sc.read_h5ad(atac_path)
print(f"ATAC: {adata_atac.shape}")

# Join scrublet results to RNA
scrublet_df = pd.read_csv(scrublet_path, index_col=0)
scrublet_df["predicted_doublet"] = scrublet_df["predicted_doublet"].map({"True": True, "False": False})
scrublet_df["doublet_score"] = scrublet_df["doublet_score"].astype("float32")
adata_rna.obs = adata_rna.obs.join(scrublet_df, how="left")
print(f"Scrublet joined: {adata_rna.obs['doublet_score'].notna().sum()}/{adata_rna.n_obs}")

# Ensure counts in X
if "counts" in adata_rna.layers:
    adata_rna.X = adata_rna.layers["counts"]
if "counts" in adata_atac.layers:
    adata_atac.X = adata_atac.layers["counts"]

print(f"\nDatasets: {adata_rna.obs['dataset'].nunique()}")
print(f"Batches: {adata_rna.obs['batch'].nunique()}")

## QC summary

In [ ]:
# Per-dataset cell counts
for ds in sorted(adata_rna.obs["dataset"].unique()):
    mask_rna = adata_rna.obs["dataset"] == ds
    n_rna = mask_rna.sum()
    # Check ATAC overlap
    rna_cells = set(adata_rna.obs_names[mask_rna])
    atac_cells = set(adata_atac.obs_names)
    n_both = len(rna_cells & atac_cells)
    print(f"{ds}: {n_rna:,} RNA cells, {n_both:,} with ATAC")

# Total overlap
shared_cells = set(adata_rna.obs_names) & set(adata_atac.obs_names)
print(f"\nTotal shared cells: {len(shared_cells):,}")

## Cell filtering

In [ ]:
# Compute QC metrics for RNA
if "total_counts" not in adata_rna.obs.columns:
    total_counts = np.array(adata_rna.X.sum(1)).squeeze()
    adata_rna.obs["total_counts"] = total_counts
if "n_genes" not in adata_rna.obs.columns:
    n_genes = np.array((adata_rna.X > 0).sum(1)).squeeze()
    adata_rna.obs["n_genes"] = n_genes

# Compute ATAC QC
if "total_counts" not in adata_atac.obs.columns:
    atac_counts = np.array(adata_atac.X.sum(1)).squeeze()
    adata_atac.obs["total_counts"] = atac_counts

# RNA filters
rna_mask = (
    (adata_rna.obs["n_genes"] > 500)
    & (adata_rna.obs["total_counts"] > 1000)
    & (adata_rna.obs["total_counts"] < 80000)
    & (adata_rna.obs["n_genes"] < 10000)
    & (adata_rna.obs["mt_frac"] < 0.20)
)
if "doublet_score" in adata_rna.obs.columns:
    has_score = adata_rna.obs["doublet_score"].notna()
    doublet_pass = ~has_score | (adata_rna.obs["doublet_score"] < 0.18)
    rna_mask = rna_mask & doublet_pass

# ATAC filters
atac_mask = (adata_atac.obs["total_counts"] > 2000) & (adata_atac.obs["total_counts"] < 100000)

# Intersect cells passing both filters
rna_pass = set(adata_rna.obs_names[rna_mask])
atac_pass = set(adata_atac.obs_names[atac_mask])
keep_cells = sorted(rna_pass & atac_pass)

n_rna_before = adata_rna.n_obs
n_atac_before = adata_atac.n_obs

adata_rna = adata_rna[keep_cells, :].copy()
adata_atac = adata_atac[keep_cells, :].copy()

print(f"RNA: {n_rna_before:,} -> {adata_rna.n_obs:,}")
print(f"ATAC: {n_atac_before:,} -> {adata_atac.n_obs:,}")
print(f"Shared filtered cells: {len(keep_cells):,}")

## Feature selection

In [ ]:
from regularizedvi.utils import filter_genes

# RNA genes
selected_genes = filter_genes(
    adata_rna,
    cell_count_cutoff=15,
    cell_percentage_cutoff2=0.01,
    nonz_mean_cutoff=1.04,
)
adata_rna = adata_rna[:, selected_genes].copy()
print(f"Selected {adata_rna.n_vars:,} RNA genes")

# ATAC peaks/tiles
selected_peaks = filter_genes(
    adata_atac,
    cell_count_cutoff=15,
    cell_percentage_cutoff2=0.01,
    nonz_mean_cutoff=1.04,
)
adata_atac = adata_atac[:, selected_peaks].copy()
print(f"Selected {adata_atac.n_vars:,} ATAC tiles")

In [ ]:
mdata = mu.MuData({"rna": adata_rna, "atac": adata_atac})
print(mdata)
print(f"\nRNA: {mdata.mod['rna'].shape}")
print(f"ATAC: {mdata.mod['atac'].shape}")

## Model parameters

In [ ]:
# Papermill parameters
results_folder = "results/immune_integration_mm/"
additive_bg_prior_alpha = 1.0
additive_bg_prior_beta = 100.0
regularise_background = 0
compute_pearson = 1
use_feature_scaling = 1
library_log_means_centering_sensitivity_rna = 1.0
library_log_means_centering_sensitivity_atac = 0.2
library_log_vars_weight = 0.2
learnable_modality_scaling = 0
modality_scale_prior_concentration = 5.0
modality_lr_multiplier_atac = None
stratify_validation_key = "harmonized_annotation+batch"
early_stopping_min_delta_per_feature = 0.0002
skip_training = 0
max_epochs = 2000
wandb_project = None
wandb_name = None
wandb_entity = None
wandb_notes = None
wandb_group = None

In [ ]:
from regularizedvi.utils import coerce_papermill_params, finish_wandb, log_figure_to_wandb, setup_wandb_logger

params = coerce_papermill_params(
    additive_bg_prior_alpha=(additive_bg_prior_alpha, float),
    additive_bg_prior_beta=(additive_bg_prior_beta, float),
    regularise_background=(regularise_background, bool),
    compute_pearson=(compute_pearson, bool),
    use_feature_scaling=(use_feature_scaling, bool),
    library_log_means_centering_sensitivity_rna=(library_log_means_centering_sensitivity_rna, "float_or_none"),
    library_log_means_centering_sensitivity_atac=(library_log_means_centering_sensitivity_atac, "float_or_none"),
    library_log_vars_weight=(library_log_vars_weight, float),
    learnable_modality_scaling=(learnable_modality_scaling, bool),
    modality_scale_prior_concentration=(modality_scale_prior_concentration, float),
    modality_lr_multiplier_atac=(modality_lr_multiplier_atac, "float_or_none"),
    stratify_validation_key=(stratify_validation_key, "str_or_none"),
    early_stopping_min_delta_per_feature=(early_stopping_min_delta_per_feature, "float_or_none"),
    skip_training=(skip_training, bool),
    max_epochs=(max_epochs, int),
    wandb_project=(wandb_project, "str_or_none"),
    wandb_name=(wandb_name, "str_or_none"),
    wandb_entity=(wandb_entity, "str_or_none"),
    wandb_notes=(wandb_notes, "str_or_none"),
    wandb_group=(wandb_group, "str_or_none"),
)
additive_bg_prior_alpha = params["additive_bg_prior_alpha"]
additive_bg_prior_beta = params["additive_bg_prior_beta"]
regularise_background = params["regularise_background"]
compute_pearson = params["compute_pearson"]
use_feature_scaling = params["use_feature_scaling"]
library_log_means_centering_sensitivity_rna = params["library_log_means_centering_sensitivity_rna"]
library_log_means_centering_sensitivity_atac = params["library_log_means_centering_sensitivity_atac"]
library_log_vars_weight = params["library_log_vars_weight"]
learnable_modality_scaling = params["learnable_modality_scaling"]
modality_scale_prior_concentration = params["modality_scale_prior_concentration"]
modality_lr_multiplier_atac = params["modality_lr_multiplier_atac"]
stratify_validation_key = params["stratify_validation_key"]
early_stopping_min_delta_per_feature = params["early_stopping_min_delta_per_feature"]
skip_training = params["skip_training"]
max_epochs = params["max_epochs"]
wandb_project = params["wandb_project"]
wandb_name = params["wandb_name"]
wandb_entity = params["wandb_entity"]
wandb_notes = params["wandb_notes"]
wandb_group = params["wandb_group"]

_sens = {}
if library_log_means_centering_sensitivity_rna is not None:
    _sens["rna"] = library_log_means_centering_sensitivity_rna
if library_log_means_centering_sensitivity_atac is not None:
    _sens["atac"] = library_log_means_centering_sensitivity_atac
library_log_means_centering_sensitivity = _sens or None

modality_lr_multiplier = None
if modality_lr_multiplier_atac is not None:
    modality_lr_multiplier = {"atac": modality_lr_multiplier_atac}

os.makedirs(results_folder, exist_ok=True)

wandb_loggers, wandb_run = setup_wandb_logger(
    wandb_project=wandb_project,
    wandb_name=wandb_name,
    wandb_entity=wandb_entity,
    wandb_notes=wandb_notes,
    wandb_group=wandb_group,
    config={
        "additive_bg_prior_alpha": additive_bg_prior_alpha,
        "additive_bg_prior_beta": additive_bg_prior_beta,
        "regularise_background": regularise_background,
        "compute_pearson": compute_pearson,
        "use_feature_scaling": use_feature_scaling,
        "library_log_means_centering_sensitivity": library_log_means_centering_sensitivity,
        "library_log_vars_weight": library_log_vars_weight,
        "learnable_modality_scaling": learnable_modality_scaling,
        "modality_lr_multiplier": modality_lr_multiplier,
        "stratify_validation_key": stratify_validation_key,
        "early_stopping_min_delta_per_feature": early_stopping_min_delta_per_feature,
        "max_epochs": max_epochs,
        "n_hidden": {"rna": 512, "atac": 256},
        "n_latent": {"rna": 128, "atac": 64},
        "n_layers": 1,
        "latent_mode": "concatenation",
    },
    results_folder=results_folder,
)

## Model setup and training

In [ ]:
regularizedvi.RegularizedMultimodalVI.setup_mudata(
    mdata,
    ambient_covariate_keys=["batch"],
    nn_conditioning_covariate_keys=["dataset", "donor"],
    feature_scaling_covariate_keys=["dataset", "donor"],
    dispersion_key="batch",
    library_size_key="batch",
    encoder_covariate_keys=False,
)

In [ ]:
model = regularizedvi.RegularizedMultimodalVI(
    mdata,
    n_hidden={"rna": 512, "atac": 256},
    n_latent={"rna": 128, "atac": 64},
    n_layers=1,
    latent_mode="concatenation",
    additive_background_modalities=["rna"],
    feature_scaling_modalities=["rna", "atac"] if use_feature_scaling else [],
    dispersion="gene-batch",
    regularise_dispersion=True,
    additive_bg_prior_alpha=additive_bg_prior_alpha,
    additive_bg_prior_beta=additive_bg_prior_beta,
    regularise_background=regularise_background,
    library_log_means_centering_sensitivity=library_log_means_centering_sensitivity,
    library_log_vars_weight=library_log_vars_weight,
    learnable_modality_scaling=learnable_modality_scaling,
    modality_scale_prior_concentration=modality_scale_prior_concentration,
    compute_pearson=compute_pearson,
)

print(model)
print(f"\nTotal latent dim: {model.module.total_latent_dim}")
print(f"  RNA Z dim: {model.module.n_latent_dict['rna']}")
print(f"  ATAC Z dim: {model.module.n_latent_dict['atac']}")
print(f"Regularise background: {regularise_background}")
print(f"Library centering: {library_log_means_centering_sensitivity}")
print(f"Library log vars weight: {library_log_vars_weight}")

In [ ]:
import time

if skip_training:
    ref_run_name = f"{results_folder}/model"
    model = regularizedvi.RegularizedMultimodalVI.load(ref_run_name, adata=mdata)
    print(f"Loaded model from {ref_run_name}")
    n_epochs = len(model.history_["elbo_train"])
else:
    from scvi.train import SaveCheckpoint

    checkpoint_cb = SaveCheckpoint(
        dirpath=f"{results_folder}/checkpoints",
        every_n_epochs=200,
        save_top_k=-1,
        filename="epoch-{epoch}",
    )

    _es_kwargs = {}
    if early_stopping_min_delta_per_feature is not None:
        _es_kwargs["early_stopping_min_delta_per_feature"] = early_stopping_min_delta_per_feature

    _plan_kwargs = {}
    if modality_lr_multiplier is not None:
        _plan_kwargs["modality_lr_multiplier"] = modality_lr_multiplier

    _ds_kwargs = {}
    if stratify_validation_key is not None:
        from sklearn.model_selection import train_test_split

        obs_df = mdata.mod[list(mdata.mod.keys())[0]].obs
        keys = stratify_validation_key.split("+")
        strat_labels = obs_df[keys[0]].astype(str)
        for k in keys[1:]:
            strat_labels = strat_labels + "_" + obs_df[k].astype(str)
        counts = strat_labels.value_counts()
        rare = counts[counts < 2].index
        if len(rare) > 0:
            strat_labels = strat_labels.replace(rare, "_rare_")
            print(f"Merged {len(rare)} rare strata into '_rare_'")
        all_idx = np.arange(mdata.n_obs)
        train_idx, val_idx = train_test_split(all_idx, test_size=0.1, stratify=strat_labels.values, random_state=42)
        _ds_kwargs["external_indexing"] = [train_idx, val_idx]
        print(f"Stratified split by {stratify_validation_key}: {len(train_idx)} train, {len(val_idx)} val")

    t0 = time.time()
    model.train(
        check_val_every_n_epoch=1,
        train_size=0.9,
        max_epochs=max_epochs,
        batch_size=1024,
        early_stopping=True,
        early_stopping_patience=20,
        early_stopping_monitor="elbo_validation",
        enable_checkpointing=True,
        callbacks=[checkpoint_cb],
        logger=wandb_loggers,
        datasplitter_kwargs=_ds_kwargs if _ds_kwargs else None,
        plan_kwargs=_plan_kwargs if _plan_kwargs else None,
        **_es_kwargs,
    )
    elapsed = time.time() - t0
    n_epochs = len(model.history_["elbo_train"])
    print(f"\nTraining: {n_epochs} epochs in {elapsed / 60:.1f} min ({elapsed / n_epochs:.2f} s/epoch)")

## Results

In [ ]:
fig = model.plot_training_diagnostics(skip_epochs=min(80, n_epochs // 2))
log_figure_to_wandb("training_diagnostics", fig)
plt.show()

In [ ]:
ref_run_name = f"{results_folder}/model"
model.save(ref_run_name, overwrite=True)

In [ ]:
model.compute_latent_umap(adata_rna)
print("Computed joint + per-modality latent UMAPs")

In [ ]:
fig = model.plot_umap_comparison(
    adata_rna,
    color=["dataset", "batch", "harmonized_annotation", "level_1"],
    umap_keys=[
        ("X_umap_joint", "Joint Z"),
        ("X_umap_rna", "RNA Z"),
        ("X_umap_atac", "ATAC Z"),
    ],
)
log_figure_to_wandb("umap_3x4", fig)
plt.show()

In [ ]:
attribution, fig_bar = model.plot_modality_attribution(batch_size=256)
log_figure_to_wandb("attribution_bar", fig_bar)
plt.show()

In [ ]:
attribution = model.store_attribution_results(adata_rna, attribution=attribution)
model.compute_attribution_umap(adata_rna)

fig = model.plot_umap_comparison(
    adata_rna,
    color=["batch", "harmonized_annotation"],
    umap_keys=[
        ("X_umap_joint", "Joint Z"),
        ("X_umap_rna", "RNA Z"),
        ("X_umap_atac", "ATAC Z"),
        ("X_umap_attr_atac", "ATAC attribution Z"),
        ("X_umap_attr_rna", "RNA attribution Z"),
    ],
    figsize_per_panel=(5, 5),
)
log_figure_to_wandb("umap_summary_5x2", fig)
plt.show()

In [ ]:
output_dir = f"{ref_run_name}/outputs/"
saved = model.save_analysis_outputs(
    output_dir,
    adata_rna,
    save_attribution=True,
    attribution=attribution,
)
print(f"Saved {len(saved)} files to {output_dir}")

In [ ]:
finish_wandb()